# Adaptive-Compute Transformer — Baseline Training (Colab)

Runs the baseline static-depth transformer from `adaptive-compute-transformer/src`.
Built module by module: Setup → Code → Data → W&B → Train → Results.
Runtime > Change runtime type > GPU (T4) before running.

## Module 1 — Setup

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected — set Runtime > Change runtime type > GPU"
print("GPU:", torch.cuda.get_device_name(0))

!pip install -q tiktoken datasets wandb

import random, numpy as np
SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print("seed set:", SEED)

## Module 2 — Code

Materializes `src/` from the repo (config.py, model.py, data.py, train.py) into the Colab filesystem, verbatim, one `%%writefile` cell per file.

In [ ]:
%%writefile config.py
"""Config for the baseline static-depth transformer.

Every training run resolves one of these to a dict and dumps it to
experiments/<run_name>/config.json before training starts, so no result
is ever reported without the exact config + seed that produced it.
"""
from dataclasses import dataclass, asdict


@dataclass
class BaselineConfig:
    # architecture
    vocab_size: int = 50257  # gpt2 tiktoken encoding
    block_size: int = 256    # context length
    n_layer: int = 12
    n_head: int = 6
    d_model: int = 384
    d_ff: int = 1536         # 4 * d_model
    dropout: float = 0.1
    bias: bool = True

    # data
    dataset_name: str = "Salesforce/wikitext"
    dataset_config: str = "wikitext-103-raw-v1"

    # optimization
    batch_size: int = 32
    grad_accum_steps: int = 1
    learning_rate: float = 3e-4
    min_lr: float = 3e-5
    weight_decay: float = 0.1
    beta1: float = 0.9
    beta2: float = 0.95
    grad_clip: float = 1.0
    warmup_steps: int = 500
    max_steps: int = 20000
    eval_interval: int = 500
    eval_iters: int = 100

    # run control
    seed: int = 1337
    run_name: str = "baseline_v1"
    out_dir: str = "experiments"
    device: str = "cuda"
    dtype: str = "bfloat16"  # falls back to float32 in train.py if unsupported
    compile: bool = True
    wandb_project: str = "adaptive-compute-transformer"
    wandb_mode: str = "offline"  # "online" once an API key is configured

    def to_dict(self):
        return asdict(self)


In [ ]:
%%writefile model.py
"""Baseline static-depth decoder-only transformer.

Standard GPT-style architecture (pre-LayerNorm, causal self-attention),
deliberately unremarkable: this is the control model. Later, the
adaptive-depth model will subclass / wrap this same block structure so
that any measured difference is attributable to routing, not to the
base architecture.
"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.d_model % cfg.n_head == 0
        self.n_head = cfg.n_head
        self.d_model = cfg.d_model
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=cfg.bias)
        self.proj = nn.Linear(cfg.d_model, cfg.d_model, bias=cfg.bias)
        self.attn_dropout = cfg.dropout
        self.resid_dropout = nn.Dropout(cfg.dropout)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(self.d_model, dim=2)
        head_dim = C // self.n_head
        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)

        y = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.attn_dropout if self.training else 0.0,
            is_causal=True,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.proj(y))


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.dropout = nn.Dropout(cfg.dropout)
        self.act = nn.GELU()

    def forward(self, x):
        return self.dropout(self.proj(self.act(self.fc(x))))


class Block(nn.Module):
    """One transformer block. Kept as a standalone module (rather than
    inlined in the stack) because the adaptive-depth model will need to
    call blocks individually, per-token, between router decisions."""

    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.mlp = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class BaselineTransformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        # weight tying
        self.tok_emb.weight = self.head.weight

        self.apply(self._init_weights)
        # scaled init for residual projections, per GPT-2 paper
        for name, p in self.named_parameters():
            if name.endswith("proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg.block_size, (
            f"sequence length {T} exceeds block_size {self.cfg.block_size}"
        )
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1
            )
        return logits, loss

    def num_params(self, exclude_embeddings=True):
        n = sum(p.numel() for p in self.parameters())
        if exclude_embeddings:
            n -= self.pos_emb.weight.numel()
        return n

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.cfg.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


In [ ]:
%%writefile data.py
"""WikiText-103 loading + tokenization + batching.

Uses HF `datasets` only for corpus I/O (streaming/download of raw text).
Tokenization (tiktoken) and batching are hand-written — no HF Trainer,
no HF data collators.
"""
import os
import numpy as np
import tiktoken

CACHE_DIR = os.path.join(os.getcwd(), "data", "tokenized")


def _tokenize_split(split, dataset_config, enc):
    from datasets import load_dataset

    ds = load_dataset("Salesforce/wikitext", dataset_config, split=split)
    eot = enc.eot_token
    ids = []
    for row in ds:
        text = row["text"]
        if not text.strip():
            continue
        ids.extend(enc.encode_ordinary(text))
        ids.append(eot)
    return np.array(ids, dtype=np.uint16)


def prepare_dataset(dataset_config="wikitext-103-raw-v1", cache_dir=CACHE_DIR):
    """Tokenizes train/validation/test splits once and caches them as
    .bin files of uint16 token ids. Safe to call every run — skips work
    if the cache already exists."""
    os.makedirs(cache_dir, exist_ok=True)
    enc = tiktoken.get_encoding("gpt2")

    paths = {}
    for split, hf_split in [("train", "train"), ("val", "validation"), ("test", "test")]:
        path = os.path.join(cache_dir, f"{split}.bin")
        paths[split] = path
        if os.path.exists(path):
            continue
        ids = _tokenize_split(hf_split, dataset_config, enc)
        ids.tofile(path)
    return paths


def get_batch(split, cfg, cache_dir=CACHE_DIR, device="cpu"):
    """Memory-maps the cached token file and samples a random batch of
    (input, target) sequences. Re-opened per call (cheap, avoids stale
    mmap handles) — standard nanoGPT-style pattern."""
    import torch

    path = os.path.join(cache_dir, f"{split}.bin")
    data = np.memmap(path, dtype=np.uint16, mode="r")

    ix = np.random.randint(0, len(data) - cfg.block_size - 1, size=cfg.batch_size)
    x = torch.stack(
        [torch.from_numpy(data[i:i + cfg.block_size].astype(np.int64)) for i in ix]
    )
    y = torch.stack(
        [torch.from_numpy(data[i + 1:i + 1 + cfg.block_size].astype(np.int64)) for i in ix]
    )
    if device != "cpu":
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    return x, y


In [ ]:
%%writefile train.py
"""Hand-written training loop for the baseline static-depth transformer.

Every run:
  1. resolves a BaselineConfig,
  2. dumps the full config + seed + git commit to experiments/<run_name>/config.json
     BEFORE training starts,
  3. logs train/val loss to W&B (offline mode by default) and to a local
     log file,
  4. checkpoints periodically.

No result from this script should ever be quoted without the matching
config.json sitting next to it.
"""
import os
import json
import time
import subprocess
import math

import torch

from config import BaselineConfig
from model import BaselineTransformer
from data import prepare_dataset, get_batch


def get_git_commit():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=os.path.dirname(__file__)
        ).decode().strip()
    except Exception:
        return "unknown"


def get_lr(step, cfg):
    if step < cfg.warmup_steps:
        return cfg.learning_rate * (step + 1) / cfg.warmup_steps
    if step > cfg.max_steps:
        return cfg.min_lr
    decay_ratio = (step - cfg.warmup_steps) / (cfg.max_steps - cfg.warmup_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return cfg.min_lr + coeff * (cfg.learning_rate - cfg.min_lr)


@torch.no_grad()
def estimate_loss(model, cfg, device):
    model.eval()
    out = {}
    for split in ("train", "val"):
        losses = torch.zeros(cfg.eval_iters)
        for i in range(cfg.eval_iters):
            x, y = get_batch(split, cfg, device=device)
            _, loss = model(x, y)
            losses[i] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


def train(cfg: BaselineConfig):
    torch.manual_seed(cfg.seed)

    device = cfg.device if torch.cuda.is_available() else "cpu"
    if device == "cpu" and cfg.device == "cuda":
        print("WARNING: CUDA requested but not available — falling back to CPU. "
              "This will be very slow; use Colab/Kaggle GPU for real runs.")

    run_dir = os.path.join(cfg.out_dir, cfg.run_name)
    os.makedirs(run_dir, exist_ok=True)

    resolved = cfg.to_dict()
    resolved["git_commit"] = get_git_commit()
    resolved["resolved_device"] = device
    with open(os.path.join(run_dir, "config.json"), "w") as f:
        json.dump(resolved, f, indent=2)

    print(f"[{cfg.run_name}] preparing dataset (cached after first run)...")
    prepare_dataset(cfg.dataset_config)

    model = BaselineTransformer(cfg).to(device)
    print(f"[{cfg.run_name}] model params (excl. pos emb): {model.num_params():,}")

    if cfg.compile and device == "cuda":
        model = torch.compile(model)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.learning_rate,
        betas=(cfg.beta1, cfg.beta2),
        weight_decay=cfg.weight_decay,
    )

    use_wandb = False
    try:
        import wandb
        os.environ["WANDB_MODE"] = cfg.wandb_mode
        wandb.init(project=cfg.wandb_project, name=cfg.run_name, config=resolved)
        use_wandb = True
    except Exception as e:
        print(f"W&B not initialized ({e}); continuing without it.")

    log_path = os.path.join(run_dir, "train_log.jsonl")
    log_f = open(log_path, "a")

    t0 = time.time()
    for step in range(cfg.max_steps + 1):
        lr = get_lr(step, cfg)
        for g in optimizer.param_groups:
            g["lr"] = lr

        if step % cfg.eval_interval == 0:
            losses = estimate_loss(model, cfg, device)
            elapsed = time.time() - t0
            record = {"step": step, "lr": lr, "elapsed_s": elapsed, **losses}
            print(f"step {step}: train_loss {losses['train']:.4f} "
                  f"val_loss {losses['val']:.4f} lr {lr:.2e}")
            log_f.write(json.dumps(record) + "\n")
            log_f.flush()
            if use_wandb:
                wandb.log(record, step=step)

            ckpt_path = os.path.join(run_dir, "ckpt.pt")
            torch.save({
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "step": step,
                "config": resolved,
            }, ckpt_path)

        x, y = get_batch("train", cfg, device=device)
        for micro_step in range(cfg.grad_accum_steps):
            _, loss = model(x, y)
            loss = loss / cfg.grad_accum_steps
            loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    log_f.close()
    if use_wandb:
        wandb.finish()


if __name__ == "__main__":
    cfg = BaselineConfig()
    train(cfg)


In [ ]:
import sys
sys.path.insert(0, '.')
import importlib
import config, model, data, train
for m in (config, model, data, train):
    importlib.reload(m)
print('code modules loaded OK:', [m.__name__ for m in (config, model, data, train)])

## Module 3 — Data

Downloads WikiText-103-raw-v1 via HF `datasets` and tokenizes it with the GPT-2 BPE tokenizer (`tiktoken`), caching to `.bin` files. Only run once per Colab session — cached after first run. This step takes a few minutes (first time only).

In [ ]:
from data import prepare_dataset
from config import BaselineConfig

cfg = BaselineConfig()
paths = prepare_dataset(cfg.dataset_config)

import os
for split, path in paths.items():
    size_mb = os.path.getsize(path) / 1e6
    assert size_mb > 0, f"{split} cache is empty — tokenization failed"
    print(f"{split}: {path} ({size_mb:.1f} MB)")

## Module 4 — W&B (optional)

Logs in to Weights & Biases so runs are tracked online. Skip this cell to log offline (default `wandb_mode="offline"` in `config.py` — runs still work and log locally to `experiments/<run_name>/train_log.jsonl`, just not to the W&B dashboard).

In [ ]:
import wandb
wandb.login()  # paste your W&B API key when prompted

# flip to online now that we're logged in — edit cfg in the Train module below
print("W&B login OK. Set cfg.wandb_mode = 'online' in the train cell to log to the dashboard.")

## Module 5 — Train

**5a. Smoke run** — tiny config (200 steps, small eval) on the real GPU to confirm the loop actually runs end-to-end and loss decreases, before committing to a long run. This is still not a reported result — just a correctness check.

**5b. Full run** — the real baseline training run. Config + seed + git commit are written to `experiments/<run_name>/config.json` before training starts, per the project's reproducibility rule.

In [ ]:
from train import train
from config import BaselineConfig

smoke_cfg = BaselineConfig(
    run_name="smoke_test_colab",
    max_steps=200,
    eval_interval=50,
    eval_iters=10,
    warmup_steps=20,
    compile=False,  # skip torch.compile for a fast smoke run
    wandb_mode="offline",
)
train(smoke_cfg)
print("SMOKE RUN COMPLETE — check that train_loss decreased above before running the full config.")

In [ ]:
# Full baseline run — only proceed once the smoke run above shows decreasing loss.
full_cfg = BaselineConfig(
    run_name="baseline_v1",
    wandb_mode="online",  # change to "offline" if you skipped the W&B login cell
)
train(full_cfg)

## Module 6 — Results

Plots train/val loss from the actual `train_log.jsonl` written by the run above — nothing here is estimated. Also copies the run's `config.json` + log next to the plot so the chart always travels with the exact config/seed that produced it, per the project's reproducibility rule.

In [ ]:
import json, os
import matplotlib.pyplot as plt

run_name = full_cfg.run_name
run_dir = os.path.join(full_cfg.out_dir, run_name)
log_path = os.path.join(run_dir, "train_log.jsonl")

records = [json.loads(line) for line in open(log_path)]
steps = [r["step"] for r in records]
train_loss = [r["train"] for r in records]
val_loss = [r["val"] for r in records]

plt.figure(figsize=(7, 4))
plt.plot(steps, train_loss, label="train loss")
plt.plot(steps, val_loss, label="val loss")
plt.xlabel("step"); plt.ylabel("cross-entropy loss"); plt.legend()
plt.title(f"Baseline transformer — {run_name}")
plt.tight_layout()

results_dir = os.path.join(os.getcwd(), "results")
os.makedirs(results_dir, exist_ok=True)
plt.savefig(f"{results_dir}/{run_name}_loss_curve.png", dpi=150)
plt.show()

print(f"final train_loss={train_loss[-1]:.4f}  val_loss={val_loss[-1]:.4f}  (step {steps[-1]})")